In [4]:
import dotenv
dotenv.load_dotenv()

True

In [15]:
# Embeddings de palavras vs embeddings de sentenças
# word2vec, glove, são exemplos de embeddings de palavras tradicionais, enquanto modelos como SentenceTransformer geram embeddings de sentenças.
# Isso permite capturar o significado de frases inteiras, não apenas de palavras isoladas.

BERT e outros modelos baseados em transformers geram embeddings de sentenças/contexto, capturando o significado das palavras no contexto. 

Word2Vec e GloVe geram embeddings de palavras, capturando relações semânticas entre palavras individuais.

Modelos como SentenceTransformer permitem gerar embeddings de sentenças inteiras, úteis para tarefas de similaridade semântica e recuperação de informações.

BERT e outros modelos baseados em transformers:
-  Prós: Capturam contexto, geram embeddings de sentenças, bons para tarefas de similaridade semântica.
-  Contras: Mais pesados, requerem mais recursos computacionais.

Word2Vec e GloVe:
- Prós: Leves, rápidos, bons para relações semânticas entre palavras.
- Contras: Não capturam contexto, não geram embeddings de sentenças diretamente.

SentenceTransformer:
- Prós: Fácil de usar para embeddings de sentenças, bom desempenho em tarefas de similaridade.
- Contras: Depende de modelos baseados em transformers, pode ser pesado para grandes volumes de dados.

In [ ]:
# Embeddings pra frases
from sentence_transformers import SentenceTransformer

import os
token = os.getenv("hf_api_token")
model = SentenceTransformer('all-MiniLM-L6-v2', token=token)
model.eval() # Coloca o modelo em modo de avaliação, desativando dropout e outras camadas de treinamento.
embeddings = model.encode([
    "Gosto de programar em Python",
    "Python é uma linguagem de programação",
    "Adoro comer pizza",
    "Em Python eu gosto de programar",
    "Gosto de programar em Python"
])
print(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15636.79it/s]


[[-0.05134744  0.05111805 -0.05824094 ...  0.07015432  0.08023759
   0.0300508 ]
 [-0.02467127  0.03331061 -0.07625818 ...  0.07777654  0.11635005
  -0.01235024]
 [-0.12626976  0.02303576 -0.06810606 ...  0.01144871  0.0001323
  -0.02602001]
 [-0.05273765  0.0673216  -0.05394716 ...  0.06924232  0.07752137
   0.02016156]
 [-0.05134744  0.05111805 -0.05824094 ...  0.07015432  0.08023759
   0.0300508 ]]


In [13]:
def cosine_similarity(vec1, vec2):
    import numpy as np
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

print(cosine_similarity(embeddings[0], embeddings[1]))
print(cosine_similarity(embeddings[0], embeddings[2]))
print(cosine_similarity(embeddings[0], embeddings[3]))
print(cosine_similarity(embeddings[0], embeddings[4]))

0.69943655
0.14598103
0.9589274
1.0


In [32]:
# Script de cálculo da atenção
import numpy as np

def softmax(x):
    # Aplica a função softmax ao longo de cada linha
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def calcular_autoatencao():
    # 1\. Vetores de Entrada X (2 tokens: 'Gato', 'dorme' com d\_model = 2)
    X = np.array([
        [1.0, 0.0],  # Token 1: "Gato"
        [0.0, 1.0]   # Token 2: "dorme"
    ])

    # 2\. Matrizes de Pesos Aprendidas (W\_Q, W\_K, W\_V)
    W_Q = np.array([
        [1.0, 2.0],
        [0.0, 1.0]
    ])
    W_K = np.array([
        [0.0, 1.0],
        [1.0, 1.0]
    ])
    W_V = np.array([
        [3.0, 0.0],
        [1.0, 2.0]
    ])

    # 3\. Gerando Matrizes Q, K e V (Q = X \* W\_Q, K = X \* W\_K, V = X \* W\_V)
    Q = np.matmul(X, W_Q)
    K = np.matmul(X, W_K)
    V = np.matmul(X, W_V)

    print("=== 1. Matriz de Consulta (Q) ===")
    print(Q)
    print("\n=== 2. Matriz de Chave (K) ===")
    print(K)
    print("\n=== 3. Matriz de Valor (V) ===")
    print(V)

    # 4. Produto Escalar (Q * K^T) -> Medida de afinidade
    pontuacoes_brutas = np.matmul(Q, K.T)
    print("\n=== 4. Pontuações de Afinidade Bruta (Q * K^T) ===")
    print(pontuacoes_brutas)

    # 5. Escalonamento por sqrt(d_k)
    d_k = K.shape[1]
    pontuacoes_escalonadas = pontuacoes_brutas / np.sqrt(d_k)
    print(f"\n=== 5. Pontuações Escalonadas (sqrt({d_k})) ===")
    print(np.round(pontuacoes_escalonadas, 4))

    # 6. Pesos de Atenção via Softmax
    pesos_atencao = softmax(pontuacoes_escalonadas)
    print("\n=== 6. Pesos de Atenção (Softmax) ===")
    print(np.round(pesos_atencao, 4))

    # 7. Saída Contextualizada Final (Pesos * V)
    saida_contextual = np.matmul(pesos_atencao, V)
    print("\n=== 7. Vetores de Saída Contextualizados (Attention(Q, K, V)) ===")
    print(np.round(saida_contextual, 4))

if __name__ == "__main__":
    calcular_autoatencao()

=== 1. Matriz de Consulta (Q) ===
[[1. 2.]
 [0. 1.]]

=== 2. Matriz de Chave (K) ===
[[0. 1.]
 [1. 1.]]

=== 3. Matriz de Valor (V) ===
[[3. 0.]
 [1. 2.]]

=== 4. Pontuações de Afinidade Bruta (Q * K^T) ===
[[2. 3.]
 [1. 1.]]

=== 5. Pontuações Escalonadas (sqrt(2)) ===
[[1.4142 2.1213]
 [0.7071 0.7071]]

=== 6. Pesos de Atenção (Softmax) ===
[[0.3302 0.6698]
 [0.5    0.5   ]]

=== 7. Vetores de Saída Contextualizados (Attention(Q, K, V)) ===
[[1.6605 1.3395]
 [2.     1.    ]]
